In [1]:
import glob
import os

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

In [3]:
SITE_LIST = [
    "AU-Preston",
    "AU-SurreyHills",
    "CA-Sunset",
    "FI-Kumpula",
    "FI-Torni",
    "FR-Capitole",
    "GR-HECKOR",
    "JP-Yoyogi",
    "KR-Jungnang",
    "KR-Ochang",
    "MX-Escandon",
    "NL-Amsterdam",
    "PL-Lipowa",
    "PL-Narutowicza",
    "SG-TelokKurau06",
    "UK-KingsCollege",
    "UK-Swindon",
    "US-Baltimore",
    "US-Minneapolis1",
    "US-Minneapolis2",
    "US-WestPhoenix",
]

In [4]:
MODEL_ROOT = "/tera12/yuanhua/dongwz/point_case/JAMES"
DEFAULT_OBS_ROOT = "../../obs"
OBS_ROOT = os.environ.get("OBS_ROOT", DEFAULT_OBS_ROOT)

In [5]:
MOD_VARS = ["f_sr", "f_olrg", "f_rnet", "f_fsena", "f_lfevpa", "f_fgrnd"]
OBS_VARS = ["SWup", "LWup", "Rnet", "Qh", "Qle", "Qg"]
CLM_VARS = ["SWup_cesmlcz", "LWup_cesmlcz", "Rn_cesmlcz", "Qh_cesmlcz", "Qle_cesmlcz", "Qg"]
CLM_OBS_VARS = ["SWup_obs", "LWup_obs", "Rn_obs", "Qh_obs", "Qle_obs", "Qg"]

In [6]:
EXPERIMENTS = [
    ("slab", r"$\mathregular{Slab}$", "#C98A4B"),
    ("urb", r"$\mathregular{Urb}$", "#4C78A8"),
    ("veg", r"$\mathregular{Urb_{veg}}$", "#72B7B2"),
    ("clm5", r"$\mathregular{CLM5U}$", "#D65F5F"),
    ("ucps", "UCPs", "#8C8C8C"),
]

In [7]:
CACHE_FILE = "21_sites_metrics.csv"

In [8]:
def calculate_metrics(observed, predicted):
    observed_ = np.asarray(observed, dtype=float).flatten()
    predicted_ = np.asarray(predicted, dtype=float).flatten()
    valid = np.isfinite(observed_) & np.isfinite(predicted_)
    observed_ = observed_[valid]
    predicted_ = predicted_[valid]

    if observed_.size < 2:
        return np.nan, np.nan, np.nan, np.nan

    corr, _ = pearsonr(observed_, predicted_)
    rmse = np.sqrt(mean_squared_error(observed_, predicted_))
    mae = np.mean(np.abs(predicted_ - observed_))
    obs_sd = np.std(observed_)
    std_ratio = np.std(predicted_) / obs_sd if obs_sd != 0 else np.nan
    return corr, rmse, mae, std_ratio

In [9]:
def find_history_files(experiment, site):
    pattern = os.path.join(MODEL_ROOT, experiment, site, "history", "*.nc")
    return sorted(glob.glob(pattern))

In [10]:
def open_history_dataset(experiment, site):
    files = find_history_files(experiment, site)
    if not files:
        return None

    if len(files) == 1:
        ds = xr.open_dataset(files[0])
    else:
        datasets = [xr.open_dataset(path) for path in files]
        ds = xr.concat(datasets, dim="time", data_vars="minimal", coords="minimal", compat="override")
        ds = ds.sortby("time")
        ds.load()
        for dataset in datasets:
            dataset.close()

    if "patch" in ds.dims:
        ds = ds.isel(patch=0, drop=True)
    return ds

In [11]:
def get_obs_path(site):
    filename = f"{site}_clean_observations_v1.nc"
    candidates = [
        os.path.join(OBS_ROOT, filename),
        os.path.join(DEFAULT_OBS_ROOT, filename),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"Observation file not found for {site}. Set OBS_ROOT or place {filename} under {OBS_ROOT}."
    )

In [12]:
def to_series(values):
    if isinstance(values, xr.DataArray):
        data = values.squeeze(drop=True).values
    elif isinstance(values, pd.Series):
        data = values.to_numpy()
    else:
        data = np.asarray(values)
    return np.asarray(data, dtype=float).reshape(-1)

In [13]:
def get_model_series(ds, obs_var, mod_var):
    if ds is None:
        return np.array([], dtype=float)

    if obs_var == "Rnet":
        values = ds["f_xy_solarin"] + ds["f_xy_frl"] - ds["f_sr"] - ds["f_olrg"]
    elif obs_var == "Qg":
        values = (
            ds["f_xy_solarin"]
            + ds["f_xy_frl"]
            - ds["f_sr"]
            - ds["f_olrg"]
            - ds["f_fsena"]
            - ds["f_lfevpa"]
        )
    else:
        values = ds[mod_var]

    return to_series(values)

In [14]:
def get_observation_series(obs_ds, ref_ds, obs_var):
    if obs_var == "Rnet":
        swdown = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWdown"][:-1])
        swup = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWup"][:-1])
        values = swdown + obs_ds["LWdown"][:-1] - swup - obs_ds["LWup"][:-1]
    elif obs_var == "Qg":
        swdown = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWdown"][:-1])
        swup = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWup"][:-1])
        values = (
            swdown
            + obs_ds["LWdown"][:-1]
            - swup
            - obs_ds["LWup"][:-1]
            - obs_ds["Qh"][:-1]
            - obs_ds["Qle"][:-1]
        )
    else:
        values = obs_ds[obs_var][:-1]

    return to_series(values)

In [15]:
def build_clm_obs_alignment(clm_df, obs_ds, ref_ds):
    clm_times = pd.to_datetime(clm_df["time"])
    swdown = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWdown"][:-1])
    swup = xr.where(ref_ds["f_xy_solarin"] == 0, 0, obs_ds["SWup"][:-1])

    obs_frame = pd.DataFrame(
        {
            "time": pd.to_datetime(ref_ds["time"].values),
            "SWdown": to_series(swdown),
            "LWdown": to_series(obs_ds["LWdown"][:-1]),
            "SWup": to_series(swup),
            "LWup": to_series(obs_ds["LWup"][:-1]),
            "Qh": to_series(obs_ds["Qh"][:-1]),
            "Qle": to_series(obs_ds["Qle"][:-1]),
        }
    )
    return pd.DataFrame({"time": clm_times}).merge(obs_frame, on="time", how="left")

In [16]:
def get_clm_series(clm_df, obs_ds, ref_ds, obs_var, clm_var, clm_obs_var):
    aligned_obs = None

    if obs_var in {"Rnet", "Qg", "Qh", "Qle"}:
        aligned_obs = build_clm_obs_alignment(clm_df, obs_ds, ref_ds)

    if obs_var in {"Rnet", "Qg"}:
        clm_swup = clm_df["SWup_cesmlcz"].where(aligned_obs["SWdown"] != 0, 0)
        obs_rnet = aligned_obs["SWdown"] + aligned_obs["LWdown"] - aligned_obs["SWup"] - aligned_obs["LWup"]
        mod_rnet = aligned_obs["SWdown"] + aligned_obs["LWdown"] - clm_swup - clm_df["LWup_cesmlcz"]

        if obs_var == "Qg":
            obs_values = obs_rnet - aligned_obs["Qh"] - aligned_obs["Qle"]
            mod_values = mod_rnet - clm_df["Qh_cesmlcz"] - clm_df["Qle_cesmlcz"]
        else:
            obs_values = obs_rnet
            mod_values = mod_rnet
    elif obs_var in {"Qh", "Qle"}:
        obs_values = aligned_obs[obs_var]
        mod_values = clm_df[clm_var]
    else:
        mod_values = clm_df[clm_var]
        obs_values = clm_df[clm_obs_var]

    return to_series(obs_values), to_series(mod_values)

In [17]:
def build_results_container():
    metrics = {}
    for metric_name in ("R", "RMSE", "MAE", "STD"):
        metrics[metric_name] = {
            exp_key: np.full((len(SITE_LIST), len(OBS_VARS)), np.nan, dtype=float)
            for exp_key, _, _ in EXPERIMENTS
        }
    return metrics

In [18]:
def fill_missing_site(metrics, site_index):
    for metric_name in metrics:
        for exp_key in metrics[metric_name]:
            metrics[metric_name][exp_key][site_index, :] = np.nan

In [19]:
def metrics_to_dataframe(metrics):
    rows = []
    for site_index, site in enumerate(SITE_LIST):
        for var_index, variable in enumerate(OBS_VARS):
            for metric_name, exp_dict in metrics.items():
                row = {
                    "site": site,
                    "variable": variable,
                    "metric": metric_name,
                }
                for exp_key, _, _ in EXPERIMENTS:
                    row[exp_key] = exp_dict[exp_key][site_index, var_index]
                rows.append(row)
    return pd.DataFrame(rows)

In [20]:
def compute_metrics():
    metrics = build_results_container()
    valid_counts = {
        site: {
            exp_key: {obs_var: 0 for obs_var in OBS_VARS}
            for exp_key, _, _ in EXPERIMENTS
        }
        for site in SITE_LIST
    }

    for site_index, site in enumerate(SITE_LIST):
        print(f"Processing {site}")

        obs_path = get_obs_path(site)
        clm_path = os.path.join('../../model_output', "clm5", f"{site}.csv")

        model_datasets = {exp_key: None for exp_key, _, _ in EXPERIMENTS if exp_key != "clm5"}

        try:
            obs_ds = xr.open_dataset(obs_path)
            clm_df = pd.read_csv(clm_path)
            for exp_key in model_datasets:
                model_datasets[exp_key] = open_history_dataset(exp_key, site)

            ref_ds = next((ds for ds in model_datasets.values() if ds is not None), None)
            if ref_ds is None:
                fill_missing_site(metrics, site_index)
                continue

            for var_index, obs_var in enumerate(OBS_VARS):
                if site == "MX-Escandon" and obs_var in {"LWup", "Rnet", "Qg"}:
                    continue

                obs_values = get_observation_series(obs_ds, ref_ds, obs_var)

                for exp_key, _, _ in EXPERIMENTS:
                    if exp_key == "clm5":
                        clm_obs, clm_mod = get_clm_series(
                            clm_df,
                            obs_ds,
                            ref_ds,
                            obs_var,
                            CLM_VARS[var_index],
                            CLM_OBS_VARS[var_index],
                        )
                        valid_counts[site][exp_key][obs_var] += np.count_nonzero(
                            np.isfinite(clm_obs) & np.isfinite(clm_mod)
                        )
                        r_value, rmse, mae, std_ratio = calculate_metrics(clm_obs, clm_mod)
                    else:
                        mod_values = get_model_series(
                            model_datasets[exp_key],
                            obs_var,
                            MOD_VARS[var_index],
                        )
                        if mod_values.size == 0:
                            continue
                        valid_counts[site][exp_key][obs_var] += np.count_nonzero(
                            np.isfinite(obs_values) & np.isfinite(mod_values)
                        )
                        r_value, rmse, mae, std_ratio = calculate_metrics(obs_values, mod_values)

                    metrics["R"][exp_key][site_index, var_index] = r_value
                    metrics["RMSE"][exp_key][site_index, var_index] = rmse
                    metrics["MAE"][exp_key][site_index, var_index] = mae
                    metrics["STD"][exp_key][site_index, var_index] = std_ratio
        finally:
            for ds in model_datasets.values():
                if ds is not None:
                    ds.close()
            if "obs_ds" in locals():
                obs_ds.close()

        print(f"Valid sample counts for {site}:")
        for exp_key, exp_label, _ in EXPERIMENTS:
            counts_text = ", ".join(
                f"{obs_var}={valid_counts[site][exp_key][obs_var]}"
                for obs_var in OBS_VARS
            )
            print(f"{exp_key}: {counts_text}")

    return metrics

In [21]:
def save_metrics_cache(metrics, cache_path=CACHE_FILE):
    df = metrics_to_dataframe(metrics)
    df.to_csv(cache_path, index=False)
    print(f"Saved cache to {cache_path}")

In [22]:
def main():
    metrics = compute_metrics()
    save_metrics_cache(metrics, CACHE_FILE)

In [23]:
if __name__ == "__main__":
    main()

Processing AU-Preston
Valid sample counts for AU-Preston:
slab: SWup=8735, LWup=15117, Rnet=15068, Qh=10820, Qle=10786, Qg=8808
urb: SWup=8735, LWup=15117, Rnet=15068, Qh=10820, Qle=10786, Qg=8808
veg: SWup=8735, LWup=15117, Rnet=15068, Qh=10820, Qle=10786, Qg=8808
clm5: SWup=8735, LWup=15115, Rnet=15066, Qh=10820, Qle=10786, Qg=8808
ucps: SWup=8735, LWup=15117, Rnet=15068, Qh=10820, Qle=10786, Qg=8808
Processing AU-SurreyHills
Valid sample counts for AU-SurreyHills:
slab: SWup=3181, LWup=6561, Rnet=6386, Qh=4671, Qle=4629, Qg=4488
urb: SWup=3181, LWup=6561, Rnet=6386, Qh=4671, Qle=4629, Qg=4488
veg: SWup=3181, LWup=6561, Rnet=6386, Qh=4671, Qle=4629, Qg=4488
clm5: SWup=3179, LWup=6559, Rnet=6384, Qh=4670, Qle=4628, Qg=4487
ucps: SWup=3181, LWup=6561, Rnet=6386, Qh=4671, Qle=4629, Qg=4488
Processing CA-Sunset
Valid sample counts for CA-Sunset:
slab: SWup=48654, LWup=77053, Rnet=76216, Qh=75527, Qle=68184, Qg=59319
urb: SWup=48654, LWup=77053, Rnet=76216, Qh=75527, Qle=68184, Qg=59319
v